In [1]:
import requests
import numpy as np
import pandas as pd
from taipy import Core,Gui, DataNode
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver
from selenium.common.exceptions import NoSuchElementException,TimeoutException
import os
import time

In [29]:
def scrap(username, password):
    driver = Driver(uc=True)
    try:
        driver.get('https://viu.to/login')
        username = username
        password = password
        # Wait for login elements to be available and enter credentials
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//*[@id="signin-username"]')))
        driver.find_element(By.XPATH, '//*[@id="signin-username"]').send_keys(username)
        driver.find_element(By.XPATH, '//*[@id="signin-password"]').send_keys(password)
        driver.find_element(By.XPATH, '//*[@id="content"]/form/p[5]/input').send_keys(Keys.ENTER)

        #redirect to a page
        driver.get('https://viu.to/user/continuewatching')
        #change the option from 3 months to beginning of time

        # Wait for the dropdown to load and change the option
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//*[@id="content"]/div[1]/div[2]/select')))
        select = Select(driver.find_element(By.XPATH, '//*[@id="content"]/div[1]/div[2]/select'))
        select.select_by_visible_text('Beginning of time')

        webdriverwait = WebDriverWait(driver, 60)
        # Wait for the page to load
        webdriverwait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '#content_results > div')))
        results = driver.find_elements(By.CSS_SELECTOR, '#content_results > div')
        # Extract text data from each result element
        data = []
        for result in results:
            try:
                information = result.find_element(By.XPATH, './div[2]').text  # Use relative XPath for robustness
                data.append(information)
            except NoSuchElementException:
                continue  # If an element is not found, skip to the next  # If an element is not found, skip to the next

        v = pd.DataFrame(data, columns=['Information'])

        v[['Title','Rating','Year']] = v['Information'].str.split('\n',expand=True)

        v.drop(columns=['Rating','Year','Information'],inplace=True)
    except(NoSuchElementException,TimeoutException) as e:
        print(f"An error occurred: {e}")
    finally:
        driver.get('viu.to/logout')
        driver.quit()

In [ ]:
def mb_scrap(): #Regardless of how I scrap the time will always be 4 minutes, this might be due to how the website is designed. 
# Initialize the WebDriver
    driver = Driver(uc=True)

    try:
        driver.get('https://www.movieboxpro.app/index/login/code_login')
        print('Check the mobile application for the login code')
        time.sleep(20)  # Wait for manual login or other conditions

        driver.get('https://www.movieboxpro.app/index/index/my_box?watched=1')
        data = []
        is_first_page = True  # To track whether we are on the first page

        while True:
            try:
                # Wait for the elements to load
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '/html/body/div/div/div[6]/div[3]')))

                # Increment the div index to move to the next element
                div_index = 1
                while True:
                    try:
                        dynamic_xpath = f'/html/body/div/div/div[6]/div[3]/div[{div_index}]/a/div/div[2]/p[3]'
                        element = driver.find_element(By.XPATH, dynamic_xpath)
                        element_text = element.text

                        # Check if the element's text is already in the data list
                        if element_text not in data:
                            data.append(element_text)
                        else:
                            print(f"Duplicate found, skipping: {element_text}")
                        div_index += 1
                    except NoSuchElementException:
                        break

                # Determine which "Next" button to use
                if is_first_page:
                    # Look for the "Next" button for the first page
                    try:
                        next_button = driver.find_element(By.XPATH, '//*[@id="page"]/ul/li/a')
                        is_first_page = False  # After the first page, switch to the subsequent pages logic
                    except NoSuchElementException:
                        print("No 'Next' button found on the first page. Exiting...")
                        break
                else:
                    # Look for the "Next" button for subsequent pages
                    try:
                        next_button = driver.find_element(By.XPATH, '//*[@id="page"]/ul/li[2]/a')
                    except NoSuchElementException:
                        print("No more pages to scrape. Exiting...")
                        break

                # Click the "Next" button and wait for the page to reload
                next_button.click()
                WebDriverWait(driver, 20).until(EC.staleness_of(next_button))

            except TimeoutException:
                print('Timed out waiting for elements to load')
                break
            m= pd.DataFrame(data,columns=['Title'])
            #Drop the first row
            m = m.iloc[1:]

    except (NoSuchElementException, TimeoutException) as e:
        print(f"An error occurred: {e}")
    finally:

        driver.quit()

In [7]:
def anime():
    driver = Driver(uc=True)
    driver.get('https://myanimelist.net/animelist/Tjkill3r?status=7')
    data = []


    for i in range(2,175):
        xpath = '//*[@id="list-container"]/div[4]/div/table/tbody['+str(i)+']/tr[1]/td[4]/a'
        elements = driver.find_element(By.XPATH,xpath)
        data.append(elements.text)
    driver.quit()

    a = pd.DataFrame(data, columns=['Title'])
    return a.head()

In [8]:
anime()

c:\Users\tedjo\anaconda3\envs\frontend\Lib\subprocess.py:1127: ResourceWarning: subprocess 22592 is still running
  _warn("subprocess %s is still running" % self.pid,
C:\Users\tedjo\AppData\Roaming\Python\Python312\site-packages\pandas\_config\config.py:639: ResourceWarning: unclosed file <_io.FileIO name=5 mode='wb' closefd=True>
  cursor = cursor[p]
C:\Users\tedjo\AppData\Roaming\Python\Python312\site-packages\pandas\_config\config.py:639: ResourceWarning: unclosed file <_io.FileIO name=6 mode='rb' closefd=True>
  cursor = cursor[p]
C:\Users\tedjo\AppData\Roaming\Python\Python312\site-packages\pandas\_config\config.py:639: ResourceWarning: unclosed file <_io.FileIO name=7 mode='rb' closefd=True>
  cursor = cursor[p]


,Title
0,Food Wars! The Third Plate
1,Golgo 13
2,Hinomaru Sumo
3,Kaiju No. 8
4,Lupin III


In [9]:
def get_content_details(content_name, api_key=None):
    """
    Fetch detailed information about a movie or TV show from the TMDB API, including cast and crew details.

    Args:
    content_name (str): The name of the movie or TV show to search for.
    api_key (str): The API key for the TMDB API. If not provided, will look for an environment variable.

    Returns:
    dict: A dictionary containing detailed information about the content, including cast and crew.
    """
    # Check API key
    api_key = api_key or os.getenv('TMDB_API_KEY')
    if not api_key:
        raise ValueError("API key is required. Set it as an argument or an environment variable 'TMDB_API_KEY'.")

    # Base URL for the API
    base_url = "https://api.themoviedb.org/3"

    try:
        # Fetch genre details for TV shows and movies
        tv_genre_url = f"{base_url}/genre/tv/list?api_key={api_key}"
        tv_genre_response = requests.get(tv_genre_url)
        tv_genre_response.raise_for_status()
        tv_genre_data = tv_genre_response.json()
        tv_genres = {genre['id']: genre['name'] for genre in tv_genre_data.get('genres', [])}

        movies_genre_url = f"{base_url}/genre/movie/list?api_key={api_key}"
        movies_genre_response = requests.get(movies_genre_url)
        movies_genre_response.raise_for_status()
        movies_genre_data = movies_genre_response.json()
        movie_genres = {genre['id']: genre['name'] for genre in movies_genre_data.get('genres', [])}

        # Search for the content
        search_url = f"{base_url}/search/multi?api_key={api_key}&query={content_name}"
        search_response = requests.get(search_url)
        search_response.raise_for_status()
        search_data = search_response.json()

        # Check if the search results are not empty
        if not search_data['results']:
            return {'Error': 'No content found with the provided title.'}

        # Get the most relevant result
        first_result = search_data['results'][0]
        content_id = first_result.get('id')
        content_type = first_result.get('media_type', 'N/A')

        # Determine the appropriate genre list
        genres = tv_genres if content_type == 'tv' else movie_genres
        content_genres = [genres.get(genre_id, 'Unknown') for genre_id in first_result.get('genre_ids', [])]

        # Fetch additional details, trailers, and recommendations
        details_url = f"{base_url}/{content_type}/{content_id}?api_key={api_key}"
        trailer_url = f"{base_url}/{content_type}/{content_id}/videos?api_key={api_key}"
        reviews_url = f"{base_url}/{content_type}/{content_id}/reviews?api_key={api_key}"
        recommendations_url = f"{base_url}/{content_type}/{content_id}/recommendations?api_key={api_key}"
        credits_url = f'{base_url}/{content_type}/{content_id}/credits?api_key={api_key}'  # Credits endpoint for cast and crew

        details_response = requests.get(details_url)
        trailer_response = requests.get(trailer_url)
        reviews_response = requests.get(reviews_url)
        recommendations_response = requests.get(recommendations_url)
        credits_response = requests.get(credits_url)

        details_response.raise_for_status()
        trailer_response.raise_for_status()
        reviews_response.raise_for_status()
        recommendations_response.raise_for_status()
        credits_response.raise_for_status()

        # Process additional data
        content_infos = details_response.json()
        trailer_data = trailer_response.json()
        reviews_data = reviews_response.json()
        recommendations_data = recommendations_response.json()
        credits_data = credits_response.json()

        # Extract cast and crew information
        cast_info = [
            {'Name': cast_member.get('name'), 'Character': cast_member.get('character'), 'Profile Path': cast_member.get('profile_path')}
            for cast_member in credits_data.get('cast', [])[:10]  # Limit to top 10 cast members
        ]

        crew_info = [
            {'Name': crew_member.get('name'), 'Job': crew_member.get('job'), 'Department': crew_member.get('department'),'Profile Path': crew_member.get('profile_path')}
            for crew_member in credits_data.get('crew', []) if crew_member.get('department') in ['Directing', 'Writing', 'Production']
        ]

        # Find the most relevant trailer
        trailer_key = next((trailer.get('key') for trailer in trailer_data.get('results', []) if trailer.get('type') == 'Trailer' and 'official' in trailer.get('name', '').lower()), 'N/A')
        trailer_link = f"https://www.youtube.com/watch?v={trailer_key}" if trailer_key != 'N/A' else 'N/A'

        # Extract reviews
        top_reviews = [review.get('content', 'No content') for review in reviews_data.get('results', [])[:10]]

        # Extract recommendations
        recommendations_info = [{
            'Title': rec.get('name') if rec.get('media_type') == 'tv' else rec.get('title'),
            'Poster': rec.get('poster_path'),
            'Content ID': rec.get('id')
        } for rec in recommendations_data.get('results', [])]

        # Determine runtime and origin country
        if content_type == 'tv':
            runtime = content_infos.get('episode_run_time', [])
            runtime = runtime[0] if runtime else None  # Get the first runtime if available
            origin_country = content_infos.get('origin_country', ['N/A'])[0]  # Get the first origin country
            num_episodes = content_infos.get('number_of_episodes', None)
        else:
            runtime = content_infos.get('runtime', 'N/A')
            origin_country = next((company.get('origin_country') for company in content_infos.get('production_companies', [])), 'N/A')
            num_episodes = 1  # For movies, set the number of episodes to 1

        # Construct the final content details
        content_info = {
            'Title': first_result.get('name') if content_type == 'tv' else first_result.get('title'),
            'Content ID': content_id,
            'Type': content_type,
            'Poster': first_result.get('poster_path'),
            'Backdrop': first_result.get('backdrop_path'),
            'Overview': first_result.get('overview'),
            'Trailer': trailer_link,
            'Genres': ', '.join(content_genres),
            'Runtime (minutes)': runtime,
            'Number of Episodes': num_episodes,
            'Rating': round(first_result.get('vote_average', 0), 2),
            'Release Year': first_result.get('first_air_date' if content_type == 'tv' else 'release_date', 'N/A'),
            'Origin Country': origin_country,
            'Reviews': top_reviews,
            'Recommendations': recommendations_info,
            'Cast': cast_info,
            'Crew': crew_info
        }

        return content_info


    except requests.exceptions.RequestException as e:
        return {'Error': f"Failed to fetch content details: {e}"}


In [19]:
data = get_content_details('Dune Part 2', "9121de210a58fbaa7cd5f0654a7ec8d9")

data

{'Title': 'Dune: Part Two',
 'Content ID': 693134,
 'Type': 'movie',
 'Poster': '/1pdfLvkbY9ohJlCjQH2CZjjYVvJ.jpg',
 'Backdrop': '/xOMo8BRK7PfcJv9JCnx7s5hj0PX.jpg',
 'Overview': 'Follow the mythic journey of Paul Atreides as he unites with Chani and the Fremen while on a path of revenge against the conspirators who destroyed his family. Facing a choice between the love of his life and the fate of the known universe, Paul endeavors to prevent a terrible future only he can foresee.',
 'Trailer': 'https://www.youtube.com/watch?v=U2Qp5pL3ovA',
 'Genres': 'Science Fiction, Adventure',
 'Runtime (minutes)': 167,
 'Number of Episodes': 1,
 'Rating': 8.16,
 'Release Year': '2024-02-27',
 'Origin Country': 'US',
 'Reviews': ['FULL SPOILER-FREE REVIEW @ https://talkingfilms.net/dune-part-two-review-the-new-generational-epitome-of-sci-fi-epics/\r\n\r\n"Dune: Part Two surpasses even the highest expectations, establishing itself as an unquestionable technical masterpiece of blockbuster filmmaking.\

In [18]:
data = pd.DataFrame.from_dict(data, orient='index').T
data

,Title,Content ID,Type,Poster,Backdrop,Overview,Trailer,Genres,Runtime (minutes),Number of Episodes,Rating,Release Year,Origin Country,Reviews,Recommendations,Cast,Crew
0,Dune: Part Two,693134,movie,/1pdfLvkbY9ohJlCjQH2CZjjYVvJ.jpg,/xOMo8BRK7PfcJv9JCnx7s5hj0PX.jpg,Follow the mythic journey of Paul Atreides as ...,https://www.youtube.com/watch?v=U2Qp5pL3ovA,"Science Fiction, Adventure",167,1,8.16,2024-02-27,US,[FULL SPOILER-FREE REVIEW @ https://talkingfil...,"[{'Title': 'Dune', 'Poster': '/d5NXSklXo0qyIYk...","[{'Name': 'Timothée Chalamet', 'Character': 'P...","[{'Name': 'Francine Maisler', 'Job': 'Casting'..."
